<a href="https://colab.research.google.com/github/mrdbourke/pytorch-deep-learning/blob/main/extras/exercises/07_pytorch_experiment_tracking_exercise_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07. PyTorch Experiment Tracking Exercise Template

Welcome to the 07. PyTorch Experiment Tracking exercise template notebook.

> **Note:** There may be more than one solution to each of the exercises. This notebook only shows one possible example.

## Resources

1. These exercises/solutions are based on [section 07. PyTorch Transfer Learning](https://www.learnpytorch.io/07_pytorch_experiment_tracking/) of the Learn PyTorch for Deep Learning course by Zero to Mastery.
2. See a live [walkthrough of the solutions (errors and all) on YouTube](https://youtu.be/cO_r2FYcAjU).
3. See [other solutions on the course GitHub](https://github.com/mrdbourke/pytorch-deep-learning/tree/main/extras/solutions).

> **Note:** The first section of this notebook is dedicated to getting various helper functions and datasets used for the exercises. The exercises start at the heading "Exercise 1: ...".

### Get various imports and helper functions

We'll need to make sure we have `torch` v.1.12+ and `torchvision` v0.13+.

In [1]:
# For this notebook to run with updated APIs, we need torch 1.12+ and torchvision 0.13+
import torch
import torchvision

In [2]:
 # Make sure we have a GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [3]:
# Get regular imports 
import matplotlib.pyplot as plt
import torch
from torch import nn
from torchvision import transforms
from torchinfo import summary
from going_modular.going_modular.engine import train_test_step
from going_modular.going_modular.data_setup import create_dataloaders

/home/jojo/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
2025-11-19 16:49:07.139382: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-19 16:49:07.171597: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-19 1

### Download data

Using the same data from https://www.learnpytorch.io/07_pytorch_experiment_tracking/

In [ ]:
# Get all the data of Food101 dataset
from going_modular.going_modular.data_import import get_data
get_data(url="http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz",
         file="food_101")

Did not find data/food_101/food-101.tar.gz directory, creating one...
Detected tar file -> extracting data...


In [ ]:
import os
import shutil

root = "data/food_101/food-101"
image_dir = os.path.join(root, "images")
splits = ["train", "test"]

for split in splits:
    # Create train/test directory
    split_dir = os.path.join(root, split)
    os.makedirs(split_dir, exist_ok=True)

    # Get the path to the train/test text files
    file = os.path.join(root, "meta", f"{split}.txt")

    # Copy train/test text files
    with open(file, "r") as f:
        lines = [line.strip() for line in f]
    
    for line in lines:
        class_name, num_image = line.split("/")
        path_image_origin = os.path.join(image_dir, f"{line}.jpg")
        path_image_destination = os.path.join(split_dir, f"{line}.jpg")
        # Make dir for each class
        class_dir = os.path.join(split_dir, class_name)
        os.makedirs(class_dir, exist_ok=True)
        # Copy image from source file to dest file
        shutil.copy(path_image_origin, path_image_destination)

In [13]:
train_dir = os.path.join(root, "train")
test_dir = os.path.join(root, "test")
train_dir, test_dir

('data/food_101/food-101/train', 'data/food_101/food-101/test')

In [4]:
# Setup training directory paths
train_dir_10_percent = "data/pizza_steak_sushi/train"
train_dir_20_percent = "data/pizza_steak_sushi_20_percent/train"

# Setup testing directory paths (note: use the same test dataset for both to compare the results)
test_dir = "data/pizza_steak_sushi/test"

# Check the directories
print(f"Training directory 10%: {train_dir_10_percent}")
print(f"Training directory 20%: {train_dir_20_percent}")
print(f"Testing directory: {test_dir}")

Training directory 10%: data/pizza_steak_sushi/train
Training directory 20%: data/pizza_steak_sushi_20_percent/train
Testing directory: data/pizza_steak_sushi/test


###  Weights

In [5]:
# EfficientNet-B3
weights_efficientnet_b3 = torchvision.models.EfficientNet_B3_Weights.DEFAULT

# ConvNext-Tiny
weights_convnext_tiny = torchvision.models.ConvNeXt_Tiny_Weights.DEFAULT

# DenseNet
weights_densnet = torchvision.models.DenseNet121_Weights.DEFAULT

### Transforms

In [6]:
# Auto_transform 
auto_transform_efficientnet_b3 = weights_efficientnet_b3.transforms()
auto_transform_convnext_tiny = weights_convnext_tiny.transforms()
auto_transform_densnet = weights_densnet.transforms()

# Data augmentation
data_augmentation_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.TrivialAugmentWide()
])

# Complete transform
complete_transform_efficientnet_b3 = transforms.Compose([
    data_augmentation_transform,
    auto_transform_efficientnet_b3
])

complete_transform_convnext_tiny = transforms.Compose([
    data_augmentation_transform,
    auto_transform_convnext_tiny
])

complete_transform_densnet = transforms.Compose([
    data_augmentation_transform,
    auto_transform_densnet
])

dict_of_transforms = {"efficientnet_b3_vanilla_data": auto_transform_efficientnet_b3, 
                      "convnext_tiny_vanilla_data": auto_transform_convnext_tiny, 
                      "densnet_vanilla_data": auto_transform_densnet, 
                      "efficientnet_b3_augmented_data": complete_transform_efficientnet_b3,
                      "convnext_tiny_augmented_data": complete_transform_convnext_tiny, 
                      "densnet_augmented_data": complete_transform_densnet}

### Models

In [7]:
# model efficientnet_b3
model_efficientnet_b3 = torchvision.models.efficientnet_b3(weights=weights_efficientnet_b3)
model_efficientnet_b3 = model_efficientnet_b3.to(device)

# model convnext_tiny
model_convnext_tiny = torchvision.models.convnext_tiny(weights=weights_convnext_tiny)
model_convnext_tiny = model_convnext_tiny.to(device)

# model resnet_50
model_densnet = torchvision.models.densenet121(weights=weights_densnet)
model_densnet = model_densnet.to(device)

models = [model_efficientnet_b3, model_convnext_tiny, model_densnet]

In [8]:
model_densnet

DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu

### Turn data into DataLoaders 

In [9]:
from itertools import cycle
from going_modular.going_modular.utils import freeze_pretrained_model

BATCH_SIZE = 32
frozen = False
dataloaders = {}
for name, transform in dict_of_transforms.items():
    # Create 10% training and test DataLoaders
    train_dataloader_10_percent, test_dataloader, class_names = create_dataloaders(train_dir=train_dir_10_percent,
                                                                                   test_dir=test_dir,
                                                                                   transform=transform,
                                                                                   batch_size=BATCH_SIZE)

    # Create 20% training and test DataLoaders
    train_dataloader_20_percent, test_dataloader, class_names = create_dataloaders(train_dir=train_dir_20_percent,
                                                                                   test_dir=test_dir,
                                                                                   transform=transform,
                                                                                   batch_size=BATCH_SIZE)
    if not frozen:
        for model in models:
            freeze_pretrained_model(model, class_names, device)
        frozen = True

    dataloaders[name] = {"train_10_percent": train_dataloader_10_percent, 
                         "train_20_percent": train_dataloader_20_percent, 
                         "test": test_dataloader,
                         "model": next(cycle(models))}

# Find the number of samples/batches per dataloader (using the same test_dataloader for both experiments)
print(f"Number of batches of size {BATCH_SIZE} in 10 percent training data: {len(next(iter(dataloaders.values()))["train_10_percent"])}")
print(f"Number of batches of size {BATCH_SIZE} in 20 percent training data: {len(next(iter(dataloaders.values()))["train_20_percent"])}")
print(f"Number of batches of size {BATCH_SIZE} in testing data: {len(next(iter(dataloaders.values()))["test"])} (all experiments will use the same test set)")
print(f"Number of classes: {len(class_names)}, class names: {class_names}")

Train data:
Dataset ImageFolder
    Number of datapoints: 225
    Root location: data/pizza_steak_sushi/train
    StandardTransform
Transform: ImageClassification(
               crop_size=[300]
               resize_size=[320]
               mean=[0.485, 0.456, 0.406]
               std=[0.229, 0.224, 0.225]
               interpolation=InterpolationMode.BICUBIC
           )
Test data:
Dataset ImageFolder
    Number of datapoints: 75
    Root location: data/pizza_steak_sushi/test
    StandardTransform
Transform: ImageClassification(
               crop_size=[300]
               resize_size=[320]
               mean=[0.485, 0.456, 0.406]
               std=[0.229, 0.224, 0.225]
               interpolation=InterpolationMode.BICUBIC
           )
Train data:
Dataset ImageFolder
    Number of datapoints: 450
    Root location: data/pizza_steak_sushi_20_percent/train
    StandardTransform
Transform: ImageClassification(
               crop_size=[300]
               resize_size=[320]
      

In [10]:
print(summary(model_efficientnet_b3, input_size=(32, 3, 224, 224)))


Layer (type:depth-idx)                                  Output Shape              Param #
EfficientNet                                            [32, 3]                   --
├─Sequential: 1-1                                       [32, 1536, 7, 7]          --
│    └─Conv2dNormActivation: 2-1                        [32, 40, 112, 112]        --
│    │    └─Conv2d: 3-1                                 [32, 40, 112, 112]        (1,080)
│    │    └─BatchNorm2d: 3-2                            [32, 40, 112, 112]        (80)
│    │    └─SiLU: 3-3                                   [32, 40, 112, 112]        --
│    └─Sequential: 2-2                                  [32, 24, 112, 112]        --
│    │    └─MBConv: 3-4                                 [32, 24, 112, 112]        (2,298)
│    │    └─MBConv: 3-5                                 [32, 24, 112, 112]        (1,206)
│    └─Sequential: 2-3                                  [32, 32, 56, 56]          --
│    │    └─MBConv: 3-6                    

### Train models

In [11]:
for model_name_data_transform, dataloader in dataloaders.items():
    for train in ["train_10_percent", "train_20_percent"]:
        results = train_test_step(model=dataloader["model"],
                                  train_dataloader=dataloader[train],
                                  test_dataloader=dataloader["test"],
                                  loss_fn=nn.CrossEntropyLoss(),    
                                  optimizer=torch.optim.Adam(dataloader["model"].parameters(), lr=0.001),
                                  device=device,
                                  experiment_name=train,
                                  model_name=model_name_data_transform,
                                  epochs=5)

[INFO] Created SummaryWriter, saving to: runs/2025-11-19/train_10_percent/efficientnet_b3_vanilla_data...


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 0
-------
Looked at 0/225 samples
Train loss: 1.059 | Train accuracy: 43.75%
Test loss: 0.928 | Test accuracy: 77.46%
Epoch 1
-------
Looked at 0/225 samples
Train loss: 0.925 | Train accuracy: 67.97%
Test loss: 0.864 | Test accuracy: 79.55%
Epoch 2
-------
Looked at 0/225 samples
Train loss: 0.811 | Train accuracy: 75.78%
Test loss: 0.712 | Test accuracy: 92.80%
Epoch 3
-------
Looked at 0/225 samples
Train loss: 0.685 | Train accuracy: 88.67%
Test loss: 0.692 | Test accuracy: 84.66%
Epoch 4
-------
Looked at 0/225 samples
Train loss: 0.740 | Train accuracy: 70.31%
Test loss: 0.710 | Test accuracy: 79.45%
Train time on cuda: 17.842 seconds
[INFO] Created SummaryWriter, saving to: runs/2025-11-19/train_20_percent/efficientnet_b3_vanilla_data...


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 0
-------
Looked at 0/450 samples
Train loss: 0.574 | Train accuracy: 89.58%
Test loss: 0.479 | Test accuracy: 94.89%
Epoch 1
-------
Looked at 0/450 samples
Train loss: 0.453 | Train accuracy: 90.42%
Test loss: 0.444 | Test accuracy: 88.83%
Epoch 2
-------
Looked at 0/450 samples
Train loss: 0.409 | Train accuracy: 90.42%
Test loss: 0.396 | Test accuracy: 91.86%
Epoch 3
-------
Looked at 0/450 samples
Train loss: 0.351 | Train accuracy: 92.29%
Test loss: 0.334 | Test accuracy: 97.92%
Epoch 4
-------
Looked at 0/450 samples
Train loss: 0.332 | Train accuracy: 92.71%
Test loss: 0.328 | Test accuracy: 94.89%
Train time on cuda: 24.254 seconds
[INFO] Created SummaryWriter, saving to: runs/2025-11-19/train_10_percent/convnext_tiny_vanilla_data...


...

Epoch 0
-------
Looked at 0/450 samples
Train loss: 0.192 | Train accuracy: 93.33%
Test loss: 0.809 | Test accuracy: 89.68%
Epoch 1
-------
Looked at 0/450 samples
Train loss: 0.232 | Train accuracy: 92.92%
Test loss: 0.282 | Test accuracy: 93.84%
Epoch 2
-------
Looked at 0/450 samples
Train loss: 0.183 | Train accuracy: 94.79%
Test loss: 0.308 | Test accuracy: 87.69%
Epoch 3
-------
Looked at 0/450 samples
Train loss: 0.186 | Train accuracy: 93.33%
Test loss: 0.342 | Test accuracy: 88.64%
Epoch 4
-------
Looked at 0/450 samples
Train loss: 0.206 | Train accuracy: 93.96%
Test loss: 0.319 | Test accuracy: 88.73%
Train time on cuda: 29.450 seconds
[INFO] Created SummaryWriter, saving to: runs/2025-11-19/train_10_percent/densnet_augmented_data...


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 0
-------
Looked at 0/225 samples
Train loss: 0.149 | Train accuracy: 96.48%
Test loss: 2.061 | Test accuracy: 91.76%
Epoch 1
-------
Looked at 0/225 samples
Train loss: 0.181 | Train accuracy: 96.88%
Test loss: 0.305 | Test accuracy: 92.80%
Epoch 2
-------
Looked at 0/225 samples
Train loss: 0.206 | Train accuracy: 95.70%
Test loss: 0.197 | Test accuracy: 93.84%
Epoch 3
-------
Looked at 0/225 samples
Train loss: 0.225 | Train accuracy: 92.97%
Test loss: 0.398 | Test accuracy: 87.69%
Epoch 4
-------
Looked at 0/225 samples
Train loss: 0.402 | Train accuracy: 81.64%
Test loss: 0.241 | Test accuracy: 91.76%
Train time on cuda: 29.316 seconds
[INFO] Created SummaryWriter, saving to: runs/2025-11-19/train_20_percent/densnet_augmented_data...


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 0
-------
Looked at 0/450 samples
Train loss: 0.174 | Train accuracy: 94.38%
Test loss: 52.964 | Test accuracy: 92.80%
Epoch 1
-------
Looked at 0/450 samples
Train loss: 0.196 | Train accuracy: 92.08%
Test loss: 0.273 | Test accuracy: 92.80%
Epoch 2
-------
Looked at 0/450 samples
Train loss: 0.177 | Train accuracy: 94.58%
Test loss: 0.176 | Test accuracy: 96.88%
Epoch 3
-------
Looked at 0/450 samples
Train loss: 0.159 | Train accuracy: 95.00%
Test loss: 0.363 | Test accuracy: 88.73%
Epoch 4
-------
Looked at 0/450 samples
Train loss: 0.208 | Train accuracy: 92.08%
Test loss: 0.205 | Test accuracy: 91.76%
Train time on cuda: 32.233 seconds


### Load result on tensorboard

In [12]:
%load_ext tensorboard
%tensorboard --logdir=runs --host=localhost --port=6006

## Exercise 3. Scale up the dataset to turn FoodVision Mini into FoodVision Big using the entire [Food101 dataset from `torchvision.models`](https://pytorch.org/vision/stable/generated/torchvision.datasets.Food101.html#torchvision.datasets.Food101)
    
* You could take the best performing model from your various experiments or even the EffNetB2 feature extractor we created in this notebook and see how it goes fitting for 5 epochs on all of Food101.
* If you try more than one model, it would be good to have the model's results tracked.
* If you load the Food101 dataset from `torchvision.models`, you'll have to create PyTorch DataLoaders to use it in training.
* **Note:** Due to the larger amount of data in Food101 compared to our pizza, steak, sushi dataset, this model will take longer to train.

In [13]:
# TODO: your code